# Validasi M2 dari berkas eksperimen
Tahap rekap membaca tiga CSV, memeriksa pasangan dan split, lalu menulis enam baris ke hasil_validasi_m2.csv. E4 memakai angka publik yang sudah dibulatkan. Untuk memperoleh metrik dari model, lanjutkan ke tahap inferensi dan evaluasi dalam HANDOFF_M2.md.

In [ ]:
from pathlib import Path
import csv
import os

root = Path.cwd()
kaggle = Path('/kaggle/input').is_dir()
repo = next((p for p in (root, *root.parents) if (p / 'HANDOFF_M2.md').is_file()), root)
inputs = Path(os.environ.get('M2_INPUT', '/kaggle/input' if kaggle else repo))
output = Path(os.environ.get('M2_OUTPUT', '/kaggle/working' if kaggle else repo / 'output_m2'))
pairs = {
    'E2': ('eksperimen_e2_val.csv', ('E2_control_native_640', 'E2_native_1280')),
    'E3': ('eksperimen_mosaic_val.csv', ('E3_control_full_640', 'E3_tile384_640')),
    'E4': ('eksperimen_e4_val.csv', ('E4_full', 'E4_sahi')),
}
rows = []
for experiment, (filename, run_ids) in pairs.items():
    files = list(inputs.rglob(filename))
    if len(files) != 1:
        raise ValueError(f'{filename}: dibutuhkan satu berkas di {inputs}, ditemukan {len(files)}')
    with files[0].open(encoding='utf-8-sig', newline='') as stream:
        source = list(csv.DictReader(stream))
    for run_id in run_ids:
        matches = [r for r in source if r['id'] == run_id and r['split'] == 'val']
        if len(matches) != 1 or int(matches[0]['gambar']) != 356:
            raise ValueError(f'Periksa split, jumlah gambar, dan ID {run_id}')
        r = matches[0]
        rows.append(dict(eksperimen=experiment, id=run_id, gambar=356,
                         ap_small=r['AP-small'], recall=r['recall_conf_0.05_iou_0.5'],
                         median_ms=r['cpu_median_ms'], p95_ms=r['cpu_p95_ms']))
output.mkdir(parents=True, exist_ok=True)
with (output / 'hasil_validasi_m2.csv').open('w', newline='', encoding='utf-8') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
for control, candidate in zip(rows[::2], rows[1::2]):
    delta = float(candidate['ap_small']) - float(control['ap_small'])
    print(f"{control['eksperimen']}: {control['id']} → {candidate['id']}, perubahan AP-small {delta:+.5f}")
print(output / 'hasil_validasi_m2.csv')